# 03_pc_<agreement>_<source>_to_<target> — deterministic pipeline enforcement

Use this run-all-safe notebook to enforce approved contracts, apply deterministic quality checks, and write valid/quarantine outputs. Metadata and lineage evidence are optional.

**You edit**
- Dataset/table names and source/target kinds
- Deterministic transformation logic cell
- Target write mode

**This notebook produces**
- Contract-enforced valid output + quarantine output
- Compact handover summary
- Optional metadata/lineage evidence section


## Segment 1: Load shared config and runtime context

What this does: loads environment config and builds notebook runtime context.

Functions used: `load_fabric_config`, `validate_notebook_name`, `build_runtime_context`, `generate_run_id`.

Output: runtime context for deterministic execution.


In [ ]:
%run 00_env_config


In [ ]:
import json
from pathlib import Path
from pyspark.sql import functions as F
from fabricops_kit.environment_config import bootstrap_fabric_env, get_path, load_fabric_config
from fabricops_kit.runtime_context import assert_notebook_name_valid, build_runtime_context, generate_run_id, validate_notebook_name
from fabricops_kit.fabric_input_output import lakehouse_table_read, lakehouse_table_write, warehouse_read, warehouse_write, seed_minimal_sample_source_table
from fabricops_kit.data_contracts import extract_required_columns, get_executable_quality_rules, load_latest_approved_contract
from fabricops_kit.data_quality import run_dq_rules, split_valid_and_quarantine, validate_dq_rules


In [ ]:
USE_SAMPLE_DATA = True
ENV_NAME = "dev"
SOURCE_LAYER = "source"
TARGET_LAYER = "product"
SOURCE_KIND = "lakehouse"
TARGET_KIND = "lakehouse"
SOURCE_TABLE = "minimal_source" if USE_SAMPLE_DATA else "TODO_source_table"
TARGET_TABLE = "sample_agreement_output" if USE_SAMPLE_DATA else "TODO_target_table"
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "target_dataset"
WRITE_MODE = "overwrite"


In [ ]:
CONFIG = load_fabric_config(CONFIG)
validate_notebook_name()
assert_notebook_name_valid()
if not USE_SAMPLE_DATA:
    bootstrap_fabric_env(config=CONFIG, env=ENV_NAME, required_targets=["source", "unified", "product"])
RUN_ID = generate_run_id(prefix="pc")
runtime_context = build_runtime_context(dataset_name=DATASET_NAME, environment=ENV_NAME, source_table=SOURCE_TABLE, target_table=TARGET_TABLE, run_id=RUN_ID)


## Load approved contract and fail fast
Load approved metadata-table contract immediately after source read and validate required columns before transformation steps.


## Segment 2: Load approved contract and source data

What this does: retrieves approved contract and reads source data from lakehouse/warehouse.

Functions used: `load_latest_approved_contract`, `get_path`, `lakehouse_table_read`, `warehouse_read`.

Output: source dataframe plus approved contract constraints.


In [ ]:
USE_LOCAL_SAMPLE_METADATA = False
# Set True only when running locally without Fabric metadata tables.
if not USE_LOCAL_SAMPLE_METADATA:
    metadata_path = get_path(ENV_NAME, "metadata", config=CONFIG)
    contract = load_latest_approved_contract(
        metadata_path,
        dataset_name=DATASET_NAME,
        object_name=SOURCE_TABLE,
        contract_type="source_input",
    )
elif USE_SAMPLE_DATA:
    local_contracts = json.loads(Path("samples/end_to_end/_output/metadata/contracts.json").read_text(encoding="utf-8"))
    matches = [c for c in local_contracts if c.get("dataset_name") == DATASET_NAME and c.get("object_name") == SOURCE_TABLE and c.get("contract_type") == "source_input" and c.get("status") == "approved"]
    if not matches:
        raise ValueError("No approved sample contract found. Run 02_ex sample mode first.")
    contract = matches[-1]


In [ ]:
if SOURCE_KIND == "lakehouse":
    source_path = get_path(ENV_NAME, SOURCE_LAYER, config=CONFIG)
    if USE_SAMPLE_DATA:
        seed_minimal_sample_source_table(source_path, table_name=SOURCE_TABLE)
    df_source = lakehouse_table_read(source_path, SOURCE_TABLE)
else:
    df_source = warehouse_read(env=ENV_NAME, target=SOURCE_LAYER, schema="dbo", table=SOURCE_TABLE, config=CONFIG)


In [ ]:
required_columns = extract_required_columns(contract)
missing = sorted(set(required_columns) - set(df_source.columns))
if missing:
    raise ValueError(f"Fail fast: missing required source columns: {missing}")


In [ ]:
df_transformed = (
    df_source.withColumn("status", F.trim(F.lower(F.col("status"))))
    .withColumn("email", F.trim(F.lower(F.col("email"))))
    .withColumn("amount", F.col("amount").cast("double"))
)


## Segment 3: Validate columns, transform, and compile rules

What this does: checks required columns, applies deterministic transforms, derives executable rules from the approved contract.

Functions used: `extract_required_columns`, `get_executable_quality_rules`, `validate_dq_rules`.

Output: curated dataframe and validated contract-driven rule set.


In [ ]:
rules = get_executable_quality_rules(contract)
validate_dq_rules(rules)
if not rules and "DQ_RULES" in globals():
    rules = DQ_RULES.get(TARGET_TABLE, [])
    validate_dq_rules(rules)


In [ ]:
# Optional technical columns section omitted in minimal path.


In [ ]:
df_standard = df_transformed


In [ ]:
# Optional output-contract strictness checks can be added in project implementations.


## Segment 4: Run DQ, split outputs, and publish

What this does: runs approved DQ rules, splits valid/quarantine rows, and writes outputs.

Functions used: `run_dq_rules`, `split_valid_and_quarantine`, `lakehouse_table_write`/`warehouse_write`.

Output: persisted valid/quarantine outputs and handover summary.


In [ ]:
dq_result = run_dq_rules(df_standard, table_name=TARGET_TABLE, rules=rules, fail_on_error=False)
display(dq_result)
df_valid, df_quarantine = split_valid_and_quarantine(df_standard, rules)

if USE_SAMPLE_DATA:
    output_dir = Path("samples/end_to_end/_output")
    output_dir.mkdir(parents=True, exist_ok=True)
    df_valid.toPandas().to_csv(output_dir / f"{TARGET_TABLE}.csv", index=False)
    df_quarantine.toPandas().to_csv(output_dir / f"{TARGET_TABLE}_QUARANTINE.csv", index=False)
    metadata_evidence_location = str(output_dir / "metadata")
else:
    metadata_evidence_location = None
    if TARGET_KIND == "lakehouse":
        target_path = get_path(ENV_NAME, TARGET_LAYER, config=CONFIG)
        lakehouse_table_write(df_valid, target_path, TARGET_TABLE, mode=WRITE_MODE)
        lakehouse_table_write(df_quarantine, target_path, f"{TARGET_TABLE}_QUARANTINE", mode=WRITE_MODE)
    else:
        warehouse_write(df_valid, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, mode=WRITE_MODE, config=CONFIG)
        warehouse_write(df_quarantine, env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=f"{TARGET_TABLE}_QUARANTINE", mode=WRITE_MODE, config=CONFIG)


In [ ]:
df_written = df_valid if USE_SAMPLE_DATA else (lakehouse_table_read(get_path(ENV_NAME, TARGET_LAYER, config=CONFIG), TARGET_TABLE) if TARGET_KIND == "lakehouse" else warehouse_read(env=ENV_NAME, target=TARGET_LAYER, schema="dbo", table=TARGET_TABLE, config=CONFIG))


In [ ]:
print("Optional: profiling metadata evidence can be generated in the optional section below.")


In [ ]:
print("Optional: quality run records can be generated in the optional section below.")


## Lineage notes

Accepted lineage patterns:
- paste exported notebook source into a project implementation when deterministic scanning is needed
- keep the minimal template on manual lineage notes/handover summary


In [ ]:
print("Optional: lineage evidence can be generated in the optional section below.")


In [ ]:
# Optional: metadata evidence for audit/handover
from fabricops_kit.data_product_metadata import build_dataset_run_record, build_quality_result_records
from fabricops_kit.data_contracts import build_contract_records
from fabricops_kit.data_lineage import build_lineage_records
import json

if USE_SAMPLE_DATA:
    md = Path("samples/end_to_end/_output/metadata")
    md.mkdir(parents=True, exist_ok=True)

dq_result_records = [r.asDict(recursive=True) for r in dq_result.collect()]
dataset_run_record = build_dataset_run_record(
    run_id=RUN_ID,
    dataset_name=DATASET_NAME,
    environment=ENV_NAME,
    source_table=SOURCE_TABLE,
    target_table=TARGET_TABLE,
    status="completed",
    started_at_utc=runtime_context.get("started_at_utc"),
    row_count_source=df_source.count(),
    row_count_output=df_valid.count(),
)
quality_records = build_quality_result_records({"results": dq_result_records, "can_continue": True}, run_id=RUN_ID, dataset_name=DATASET_NAME, table_name=SOURCE_TABLE, table_stage="source")
contract_records = build_contract_records(contract)
lineage_records = build_lineage_records(dataset_name=DATASET_NAME, run_id=RUN_ID, source_tables=[SOURCE_TABLE], target_table=TARGET_TABLE, transformation_steps=[{"step": 1, "source": SOURCE_TABLE, "target": TARGET_TABLE, "operation": "Contract-driven DQ with valid/quarantine split"}])

if USE_SAMPLE_DATA:
    (md / "dataset_run.json").write_text(json.dumps(dataset_run_record, default=str, indent=2), encoding="utf-8")
    (md / "quality_records.jsonl").write_text("
".join(json.dumps(r, default=str) for r in quality_records), encoding="utf-8")
    (md / "contract_records.json").write_text(json.dumps(contract_records, default=str, indent=2), encoding="utf-8")
    (md / "lineage_records.jsonl").write_text("
".join(json.dumps(r, default=str) for r in lineage_records), encoding="utf-8")
    metadata_evidence_location = str(md)



In [ ]:
handover_summary = {
    "run_id": RUN_ID,
    "contract_id": contract.get("contract_id"),
    "contract_status": contract.get("status"),
    "source_count": df_source.count(),
    "valid_count": df_valid.count(),
    "quarantine_count": df_quarantine.count(),
    "target_table": TARGET_TABLE,
    "quarantine_table": f"{TARGET_TABLE}_QUARANTINE",
    "dq_rule_count": len(rules),
    "metadata_evidence_location": metadata_evidence_location,
}
display(handover_summary)

